In [1]:
# Imports and pip installations (if needed)
import pandas as pd

# Part 1: Load the dataset

In [ ]:
# Load the given datasets
data_ckd_cat = pd.read_csv("data/chronic_kidney_disease_categorical.csv")
data_ckd_num = pd.read_csv("data/chronic_kidney_disease_numerical.csv")

# Print the data
print("Categorical data:\n", data_ckd_cat.to_string())
print("Numerical data:\n", data_ckd_num.to_string())

# Part 2: Analyze the Dataset

Refer to this: https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease

Explain what the each data is in your own words. What are the features and labels? Are the features in the given datasets : categorical, numerical or both? Give 3 examples of categorical and numerical columns each (if they exist)

Answer:

This dataset contains data for predicting Chronic Kidney Disease. The data was collected over a close to 2 month period at a hospital. I would guess (but cannot be certain) that the data represents medical information and hospital lab values for patients who were diagnosed with Chronic Kidney Disease.

The dataset has 23 features: age, blood pressure, albumin, sugar, red blood cells, pus cell, pus cell clumps, bacteria, blood glucose random, blood urea, serum creatinine, sodium, potassium, hemoglobin, packed cell volume, white blood cell count, red blood cell count, hypertension, diabetes mellitus, coronary artery disease, appetite, pedal edema, and anemia.\
The target variable is "class" ("Target" in the csvs) with the possible labels "ckd" and "not ckd", representing Chronic Kidney Disease status.\
The dataset in the link above has a "specific gravity" feature that is missing from the data used here. Additionally, in the linked dataset, "Target" is called "class", "wbcc" (white blood cell count) is called "wc", and "rbcc" (red blood cell count) is called "rc".

Some of these features are categorical (e.g. hypertension, appetite, anemia) and others are numerical (e.g. age, blood pressure, sodium).

# Part 3: Data Preprocessing

A fundamental skill in Machine Learning is mastering the art of data cleaning and preprocessing. In this assignment, you will learn and apply essential data cleaning techniques to transform a raw dataset into a clean, ready-to-use form which you can use for regression or classification tasks. By the end of this assignment, you'll have a fully clean dataset and a solid foundation in preparing data for various machine learning models.

## Part 3.1 : Drop Duplicate rows

Let's start by checking if the given datasets have any duplicate rows (same Unique Id). Use pandas to identify and remove these duplicate rows from the given dataset

In [ ]:
# For the numerical dataset, check if there are duplicate rows in the dataset. If yes, print total number of duplicate rows
numerical_dups = data_ckd_num[data_ckd_num.duplicated()]
if numerical_dups.size > 0:
    print("Number of duplicates in the numerical dataset: ", numerical_dups.size)
    print("Duplicate rows of the numerical dataset: \n", numerical_dups)

# Drop these duplicate rows
data_ckd_num.drop_duplicates(inplace=True)

# Repeat the same for categorical dataset. Print the duplicate rows and drop them
categorical_dups = data_ckd_cat[data_ckd_cat.duplicated()]
if categorical_dups.size > 0:
    print("Number of duplicates in the categorical dataset: ", categorical_dups.size)
    print("Duplicate rows of the categorical dataset: \n", categorical_dups)

data_ckd_cat.drop_duplicates(inplace=True)

## Part 3.2: Combine two differents datasets

A good skill to have is to know how to combine 2 different datasets.

Are all the unique ids are present in both datasets? Why do you think so? If not, what do the rows that are missing from one of the datasets look like in the combined table?

Answer:



In [ ]:
# Merge the two given numerical and categorical datasets based on their unique_ID.
data_ckd_merged = pd.merge(data_ckd_cat, data_ckd_num, how="outer", on="unique_id")

#Print the combined dataset
print("Merged dataset: \n", data_ckd_merged.to_string())

## Part 3.3: Rows with Missing values

Removing missing values from a dataset is important for classification because it ensures the model is trained on complete and accurate data, leading to better performance and reliable predictions. Incomplete data can introduce bias and errors, negatively impacting the model's effectiveness.

In [ ]:
# Calculate the percentage of rows that contain atleast one missing value
def has_missing(row: pd.Series) -> bool:
    return row.isnull().any()

percent_missing = data_ckd_merged.apply(has_missing, axis=1).sum() / len(data_ckd_merged)

# Print %
print(f"Percentage of rows with missing values: {percent_missing * 100:.2f}%")

# Drop these rows from the dataset
data_ckd_nonnull = data_ckd_merged.dropna()
data_ckd_nonnull.reset_index(drop=True, inplace=True)

# Print the Dataset
print("Dataset cleaned of missing data:\n", data_ckd_nonnull.to_string())

## Part 3.4: Sort the dataset according to the Labels

In [ ]:
# Sort the dataset according to the values in 'Target' column. Make sure reset the indices after sorting
data_ckd_sorted = data_ckd_nonnull.sort_values("Target")
data_ckd_sorted.reset_index(drop=True, inplace=True)

# Print the dataset
print("Dataset sorted by target variable:\n", data_ckd_sorted.to_string())

## Part 3.5: Encoding Categorical data

In this step, we identify and process the categorical columns in the sorted dataset. We map each unique value in these columns to separate "dummy" columns.

For example, the column 'rbc' will be transformed into two columns 'rbc_normal' and 'rbc_abnormal'. If a row's value in 'rbc' is 'normal', the 'rbc_normal' column will be set to 1 and 'rbc_abnormal' will be set to 0.


**Note: Find a correct pandas function to do this **

In [ ]:
# Write code here
categorical_columns = ['rbc', 'pc', 'pcc', 'ba', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'Target']
data_ckd_dummified = pd.get_dummies(data_ckd_sorted, columns=categorical_columns)

# Print the dataset
print("Dataset with dummies:\n", data_ckd_dummified.to_string())

In the example we went through above, another solution is to have a single column for the binary variable. In the downstream modeling would this be equivalent? What effect would this have if the categorical variable could take more than 2 values? For example, let's say we have a categorical feature that is "type of condiment" that can take 5 separate values and we are trying to predict the rating of a particular sandwich.

## Part 3.6 : Remove Outliers from Numerical Columns

Outliers can disproportionately influence the fit of a regression model, potentially leading to a model that does not generalize well therefore it is important that we remove outliers from the numerical columns of the dataset.

For this dataset, we define an outlier to be 3 times the standard deviation from the mean. Drop these outliers from the dataset

In [ ]:
def find_outliers(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    means = df[cols].mean()
    stds = df[cols].std()
    
    def has_outlier(row: pd.Series) -> bool:
        for col in cols:
            if row[col] > means[col] + 3 * stds[col] or row[col] < means[col] - 3 * stds[col]:
                return True
        
        return False
    
    return df.apply(has_outlier, axis=1)

# Remove outliers
numerical_columns = ['age', 'bp', 'bgr', 'bu', 'sc', 'sod', 'pot', 'hemo', 'pcv', 'wbcc', 'rbcc']
outlier_rows = find_outliers(data_ckd_dummified, numerical_columns)
data_ckd_no_outliers = data_ckd_dummified[~outlier_rows]
data_ckd_no_outliers.reset_index(drop=True, inplace=True)

# Print the dataset
print("Dataset with outliers removed:\n", data_ckd_no_outliers.to_string())

## Part 3.7 : Normalize the Numerical Columns

Normalizing numerical attributes ensures that all features contribute equally to the model by scaling them to a consistent range, which improves model performance and convergence. It prevents features with larger scales from disproportionately influencing the model's learning process.

In [ ]:
# Normalize the all Numerical Attributes in the dataset.
data_ckd_normalized = data_ckd_no_outliers.copy()
data_ckd_normalized[numerical_columns] = (data_ckd_no_outliers[numerical_columns] - data_ckd_no_outliers[numerical_columns].mean()) / data_ckd_no_outliers[numerical_columns].std()

# Print the dataset
print("Normalized dataset:\n", data_ckd_normalized.to_string())

## Part 3.8: Remove Unnecessary columns

Are there any columns in this dataset which are not appropriate for modeling and predictions? Which column(s)? Justify their exclusion and remove them

Answer:

The unique_id column is not appropriate as a feature. It is simply an id and does not affect Chronic Kidney Disease prediction. The Target columns are necessary for the dataset because they represent the target variable. All other columns are features that are relevent due to their (I assume, as I am not a doctor) association with Chronic Kidney Disease.

In [ ]:
# Remove that column
data_ckd_results = data_ckd_normalized.drop(columns=["unique_id"])

# Print the dataset
print("Completed dataset without unnecessary columns:\n", data_ckd_results.to_string())

## Part 3.9: Export the Cleaned Data

Now that you've completed these cleaning steps you should have a pandas dataframe which is much cleaner and ready for modeling. Our final step is to save our work. Export the DataFrame to a two new formats: csv and json.

In [11]:
# Export the dataframe to a new csv file
data_ckd_results.to_csv("data_ckd_results.csv")

# Export the dataframe to a new json file
data_ckd_results.to_json("data_ckd_results.json")

# Part 4: Data conversions with Large Language Models

One powerful use case of ChatGPT (and other generative language models) is cleaning and transforming data. In some cases, these models can directly manipulate loosely structured data that you provide to them into a standard format. In the other cases, you can often prompt the model to create a conversion or extraction script for you in python or Pandas and then run it on your own. 

In this part of the assignment you will prompt 383GPT to explore these capabilities.

## Part 4.1 GPT Data Manipulation

Take the cleaned dataset that you created in part three and output the top 15 rows of that dataset. Then copy the terminal output, open 383gpt and ask it to convert that output to a markdown table. Paste that markdown table in the cell below

### Paste the markdown table here
|    |   al |   su |        age |        bp |       bgr |       bu |       sc |       sod |       pot |       hemo |       pcv |       wbcc |       rbcc |   rbc_abnormal |   rbc_normal |   pc_abnormal |   pc_normal |   pcc_notpresent |   pcc_present |   ba_notpresent |   ba_present |   htn_no |   htn_yes |   dm_no |   dm_yes |   cad_no |   cad_yes |   appet_good |   appet_poor |   pe_no |   pe_yes |   ane_no |   ane_yes |   Target_ckd |   Target_notckd |
|----:|------:|------:|-----------:|-----------:|----------:|---------:|---------:|----------:|----------:|-----------:|----------:|-----------:|-----------:|----------------:|--------------:|---------------:|------------:|------------------:|---------------:|----------------:|-------------:|---------:|----------:|-------:|---------:|---------:|----------:|--------------:|--------------:|--------:|----------:|---------:|----------:|--------------:|-----------------:|
|  0 |  2.0 |  0.0 |  0.796206  |  0.723662  |  1.137875 |  3.951101|  1.655981| -0.934242 |  1.298121 |  -2.738870 |  -2.712933|   0.506560 |  -2.204919 |          True |         False |          True |       False |             True |         False |          True |        False |    False |      True |  False |      True |     False |      True |          False |        True  |   False |      True |     True |      False |         True |             False |
|  1 |  2.0 |  2.0 |  0.921652  |  2.765149  |  3.506466 | -0.236745|  1.189801|  0.456741 | -1.277767 |  -0.549377 |  -0.528732|   0.770681 |  -1.026852 |         False |          True |         False |       True  |             True |         False |          False |        True |    False |      True |   True |      False |     True  |      False |           True |      False |   True  |      False |    True |   False |         True |             False |
|  2 |  2.0 |  0.0 |  1.548881  |  0.723662  |  2.908784 |  3.728738|  2.122160| -0.412624 |  2.207257 |  -1.582157 |  -1.484320|  -0.373843 |  -0.909046 |          True |         False |          True |       False |             True |         False |          True |        False |    False |      True |  False |      True |     False |      True |          False |        True  |   False |      True |     True |      False |         True |             False |
|  3 |  4.0 |  0.0 | -1.712709  |  1.744405  | -0.323125 | -0.051442|  0.190845| -2.672971 | -1.277767 |  -2.491003 |  -2.849446|   1.915205 |  -1.380273 |         False |          True |          True |       False |             False |          True |           False |        True |     True |      False |   True |      False |     True  |      False |          True  |      False |   True   |      False |    True |      False |         True |             False |
|  4 |  4.0 |  2.0 |  0.984375  |  2.765149  |  0.916512 |  0.467406|  3.853684| -0.064878 |  0.388984 |  -2.656248 |  -2.439908|  -0.241782 |  -1.969306 |          True |         False |          True |       False |             True |         False |           False |        True |    False |      True |   False |      True |     True  |      False |          True  |      False |   True   |      False |    True |      False |         True |             False |
|  5 |  3.0 |  1.0 |  0.733483  | -1.317826  |  3.683557 | -0.199685|  0.190845| -1.803607 | -2.035381 |  -2.656248 |  -2.576421|   3.147769 |  -2.440533 |         False |          True |          True |       False |             False |          True |           True |       False |    False |      True |   True |      False |     True  |      False |          False |       True  |    True |      False |    True |      False |         True |             False |
|  6 |  2.0 |  0.0 |  0.482592  |  1.744405  |  0.163875 |  2.431617|  3.520699| -1.629734 |  0.692029 |  -2.160514 |  -2.030370|  -0.726004 |  -1.969306 |          True |         False |          True |       False |             True |         False |           True |        False |    False |      True |   True |      False |     True  |      False |           True |        True  |  False  |      True |     True |      False |         True |             False |
|  7 |  4.0 |  3.0 |  1.297989  | -0.297082  |  2.045466 |  2.023951|  3.254310| -3.542336 | -0.671676 |  -2.036580 |  -2.166883|   1.519023 |  -2.087113 |         False |          True |          True |       False |             False |          True |           False |        True |    False |      True |  False |      True |     True  |      False |           True |       True |   False |      True |     True |      False |         True |             False |
|  8 |  1.0 |  0.0 | -0.144637  | -1.317826  |  0.916512 |  1.875708|  1.256398|  0.108995 | -0.520153 |  -1.871335 |  -2.166883|   2.883648 |  -2.204919 |         False |          True |         False |       True  |             True |         False |           True |        False |    False |      True |  False |      True |     True  |      False |          True  |        True  |   True  |      False |    True |      False |         True |             False |
|  9 |  3.0 |  0.0 |  0.858929  | -0.297082  |  0.008921 |  0.022679|  0.190845| -0.760370 |  0.540507 |  -0.714622 |  -0.665244|  -0.065702 |  -1.380273 |         False |          True |          True |       False |             True |         False |           True |        False |    False |      True |  False |      True |     True  |      False |          True  |        True  |   True  |      False |    True |      False |         True |             False |
| 10 |  3.0 |  1.0 |  0.419869  |  0.723662  |  2.045466 |  1.171557|  1.655981| -0.586497 |  0.843552 |  -1.416912 |  -1.347807|  -0.285802 |  -1.615886 |         False |          True |          True |       False |             False |          True |           False |        True |    False |      True |  False |      True |     True  |      False |          True  |        True  |  False  |      True |     True |      False |         True |             False |
| 11 |  4.0 |  1.0 |  0.984375  | -1.317826  |  2.598875 |  0.615648|  1.922369| -0.586497 |  1.601166 |  -1.995269 |  -2.030370|  -0.241782 |  -1.969306 |         True |         False |          True |       False |             True |         False |           False |        True |    False |      True |  False |      True |     True  |      False |          True  |        True  |   False |      True |     True |      False |         True |             False |
| 12 |  4.0 |  0.0 |  1.423435  | -1.317826  | -0.079625 |  3.098708|  2.588340| -0.760370 |  0.843552 |  -1.210356 |  -1.211295|   3.147769 |  -0.909046 |         False |          True |         False |       True  |             True |         False |           True |        False |    False |      True |  False |      True |     True  |      False |          True  |        True  |   False |      True |     True |      False |         True |             False |
| 13 |  3.0 |  0.0 |  1.423435  | -0.297082  |  2.156148 |  1.505102|  1.456190| -1.281988 |  0.085938 |  -1.623468 |  -1.484320|  -1.078165 |  -1.733693 |         False |          True |          True |       False |             False |          True |           False |        True |    False |      True |  False |      True |     True  |      False |           True |       True   |  False  |      True |          True |      False |         True |             False |
| 14 |  1.0 |  0.0 |  0.670760  |  0.723662  |  4.015603 | -0.236745| -0.075543| -3.194590 | -1.277767 |  -1.623468 |  -1.211295|   1.254903 |  -0.909046 |          True |         False |         False |       True  |             True |         False |           True |        False |     True |      False |  False |      True |     True  |      False |          True  |        True  |   False |      True |     True |      False |         True |             False |


** Caution: ** while language models can perform data conversions they also can * hallucinate * during this process, particularly for bigger datasets. Reflect on this below, how could you mitigate data conversion hallucinations from LLM conversions?

* Check that it displays correctly with something that displays markdown (like this notebook).
* Visually checking the data, of course, can help.
    * Make sure the right number of rows and columns exist and that they have the right names.
    * Check a good sample of random cell coordinates (or every cell if you have the time) to see if any discrepancies are present.
* Performing the same prompt in a separate chat and confirming equal results. Unless they both hallucinated the exact same way (which is possible) hallucinations will be caught.
* Prompt with careful wording such as specifying to the LLM what format the data you are giving it is in (e.g. to_string terminal output, csv, json...)
* Ask the LLM to convert in the opposite direction with some of the values changed. If the result is the same as what you input except for the values you changed, it is less likely that hallucination occurred.

## Part 4.2 GPT Pandas Prompting

In this section, you will prompt 383GPT to write pandas code manipulations for you.

After working with this data for awhile, we realized we're starting to forget the meanings of the abbreviated column names. Let's ask 383GPT to fix this for us. First, navigate to the [UCI dataset overview](https://archive.ics.uci.edu/dataset/336/chronic+kidney+disease) and copy the abbrevation to name mapping. Then, go to 383GPT and instruct the LLM to provide you with a pandas script to apply this renaming to all the columns of your dataset. Paste that code below and make any adjustments necessary to run it in your notebook.

In [ ]:
# Code to rename all the columns in the dataset
rename_mapping = {
    "age": "age",
    "bp": "blood pressure",
    "al": "albumin",
    "su": "sugar",
    "rbc_normal": "red blood cells normal",
    "rbc_abnormal": "red blood cells abnormal",
    "pc_normal": "pus cell normal",
    "pc_abnormal": "pus cell abnormal",
    "pcc_present": "pus cell clumps present",
    "pcc_notpresent": "pus cell clumps not present",
    "ba_present": "bacteria present",
    "ba_notpresent": "bacteria not present",
    "htn_yes": "hypertension yes",
    "htn_no": "hypertension no",
    "dm_yes": "diabetes mellitus yes",
    "dm_no": "diabetes mellitus no",
    "cad_yes": "coronary artery disease yes",
    "cad_no": "coronary artery disease no",
    "appet_good": "appetite good",
    "appet_poor": "appetite poor",
    "pe_yes": "pedal edema yes",
    "pe_no": "pedal edema no",
    "ane_yes": "anemia yes",
    "ane_no": "anemia no",
    "bgr": "blood glucose random",
    "bu": "blood urea",
    "sc": "serum creatinine",
    "sod": "sodium",
    "pot": "potassium",
    "hemo": "hemoglobin",
    "pcv": "packed cell volume",
    "wbcc": "white blood cell count",
    "rbcc": "red blood cell count",
    "Target_ckd": "chronic kidney disease yes",
    "Target_notckd": "chronic kidney disease no"
}

data_ckd_results.rename(columns=rename_mapping, inplace=True)

print(data_ckd_results.to_string())

## Part 4.3 Augmenting our skills with prompting

In addition, we can also use 383GPT to convert our data manipulation operations between different data manipulation languages and libraries. For example let's prompt 383GPT to convert the following SQL query to a pandas query.

**SQL Query**
```sql
SELECT Target, COUNT(*) AS count
FROM your_table_name
GROUP BY Target;
```

Prompt 383GPT to convert this to a pandas query. Run this query below, then describe what it does. (If you're not familiar with SQL that is okay you need to only comment on the final resulting output.)

In [ ]:
# Converted SQL to Pandas code
query_result = data_ckd_results.groupby('chronic kidney disease yes').size().reset_index()


query_result.columns = ['chronic kidney disease yes', 'count']

print(query_result.to_string())

*Describe what the above code does here*

(Note: As Target was split into dummy values earlier and I renamed those to "chronic kidney disease yes" and "chronic kidney disease no", I grouped by "chronic kidney disease yes" instead of "Target")

The code groups the data in the DataFrame by the possible values of "chronic kidney disease yes" (which are True and False) for aggregation by the size() aggregation function, which simply counts how many rows are in the group. It also resets the index on the DataFrame, renames the size aggregation column "count" (as the "AS count" would do), and prints to_string to the console.